In [ ]:
import pandas as pd
import geopandas as gpd
import folium
import json

# ─────────────────────────────────────────
# 1. 3개월치 로드 & 원지표 평균
# ─────────────────────────────────────────
df_list = []
for month in ['2022_06', '2022_07', '2022_08']:
    df = pd.read_csv(f'hvi_{month}.csv', dtype={'hdong_code': str})
    df_list.append(df)

df_all = pd.concat(df_list, ignore_index=True)

df_avg = df_all.groupby(['hdong_code', 'gu_name', 'hdong_name'])[
    ['low_ratio_combined', 'old_ratio', 'unit_density']
].mean().reset_index()


In [ ]:
# ─────────────────────────────────────────
# 2. 재채점 (quintile 기반, 기존 로직 동일)
# ─────────────────────────────────────────
def quintile_score(series):
    return pd.qcut(series, q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# pd.qcut(series, q=5, labels=[1,2,3,4,5], duplicates='drop').astype(int) > 오류 뜨면 수정

df_avg['score_low']     = quintile_score(df_avg['low_ratio_combined'])
df_avg['score_old']     = quintile_score(df_avg['old_ratio'])
df_avg['score_density'] = quintile_score(df_avg['unit_density'])
df_avg['HVI_score']     = df_avg['score_low'] + df_avg['score_old'] + df_avg['score_density']

# rank, grade
df_avg['HVI_rank'] = df_avg['HVI_score'].rank(ascending=False, method='min').astype(int)

def assign_grade(score):
    if score >= 13: return 'A'
    elif score >= 10: return 'B'
    elif score >= 7:  return 'C'
    elif score >= 4:  return 'D'
    else:             return 'E'

df_avg['HVI_grade'] = df_avg['HVI_score'].apply(assign_grade)

print(df_avg['HVI_score'].value_counts().sort_index())
print(df_avg[['hdong_name', 'HVI_score', 'HVI_rank', 'HVI_grade']].sort_values('HVI_rank').head(20))



In [ ]:
# ─────────────────────────────────────────
# 3. Shapefile 병합
# ─────────────────────────────────────────
gdf = gpd.read_file('seoul_dong_2021year_wgs84.shp')
gdf = gdf.rename(columns={'ADM_DR_CD': 'hdong_code', 'ADM_DR_NM': 'hdong_name'})
gdf['hdong_code'] = gdf['hdong_code'].astype(str)

gdf_merged = gdf.merge(df_avg, on='hdong_code', how='left')

# 병합 누락 확인
missing = gdf_merged[gdf_merged['HVI_score'].isna()]['hdong_name_x'].tolist()
if missing:
    print(f"[경고] 매핑 안 된 행정동 {len(missing)}개: {missing}")



In [ ]:
# ─────────────────────────────────────────
# 4. Folium Choropleth
# ─────────────────────────────────────────
m = folium.Map(location=[37.5665, 126.9780], zoom_start=11, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=json.loads(gdf_merged.to_json()),
    data=df_avg,
    columns=['hdong_code', 'HVI_score'],
    key_on='feature.properties.hdong_code',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name='HVI 주거취약지수 (3개월 평균)',
    nan_fill_color='lightgray'
).add_to(m)

# 툴팁
folium.GeoJson(
    json.loads(gdf_merged.to_json()),
    style_function=lambda x: {'fillOpacity': 0, 'weight': 0},
    tooltip=folium.GeoJsonTooltip(
        fields=['hdong_name_x', 'gu_name', 'HVI_score', 'HVI_rank', 'HVI_grade'],
        aliases=['행정동', '자치구', 'HVI 점수', '순위', '등급'],
        localize=True
    )
).add_to(m)

m.save('hvi_choropleth.html')
print("지도 저장 완료: hvi_choropleth.html")




In [ ]:
# ─────────────────────────────────────────
# 5. 모아센터 오버레이 (데이터 있을 경우)
# ─────────────────────────────────────────
# df_moa = pd.read_csv('moasenter.csv')
# for _, row in df_moa.iterrows():
#     folium.CircleMarker(
#         location=[row['LAT'], row['LNG']],
#         radius=6,
#         color='blue',
#         fill=True,
#         fill_opacity=0.9,
#         popup=row['center_name']
#     ).add_to(m)
# m.save('hvi_with_moa.html')